# Credit Cluster Scoring Tool

Use this notebook to score a private, simulated, or competitor company against the saved credit clustering model.

Expected workflow:

1. Put the saved artifact at `saved_models/credit_cluster_model_v1.joblib`.
2. Enter company financials manually, or load a CSV/Excel file.
3. Run the scoring cells.
4. Review cluster assignment, near-default affinity, feature coverage, and warning flags.

Important: the output is **not a formal credit rating** and not a calibrated probability of default. It is a public-company financial profile match based on distance to the learned KMeans credit clusters.

In [1]:
#Import your libraries here

import numpy as np
import pandas as pd
import joblib
from pathlib import Path
import sys

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

## 1. Load saved fitted model artifact

The training notebook should have saved this file after fitting the KMeans model.

In [2]:
# Import project modules and load trained artifact

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.utils import (
    build_credit_report_tables,
    save_credit_report_outputs,
    save_credit_pdf_report,
)

from src.credit_clustering import (
    # Artifact utilities
    load_artifact,
    get_segment_artifact,
    infer_near_default_cluster_from_artifact,

    # Scoring and diagnostics
    score_companies,
    add_adjacent_bucket_distances_and_outlook,
    compare_to_cluster_profile,
    make_scenarios,

    #Analyst guardrails
    apply_credit_guardrails,

    # Config
    DEFAULT_PRIMARY_SEGMENT,
    DEFAULT_SCORING_TEMPERATURE,
    DEFAULT_FX_TO_MODEL_CURRENCY,
    DEFAULT_SCORING_MIN_DENOMINATOR,
    RATIO_COLS,
    SUMMARY_COLS_WITH_OUTLOOK,
    SCENARIO_SUMMARY_COLS,
)

MODEL_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "saved_models"
    / "nonfinancial_credit_scorecard_kmeans_k5_v3.joblib"
)

if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Model artifact not found at {MODEL_PATH}")

INPUT_PATH = PROJECT_ROOT / "inputs"
INPUT_PATH.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = PROJECT_ROOT / "outputs" / "company_scores"
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

artifact = load_artifact(MODEL_PATH)

SCORING_SEGMENT = artifact.get("primary_segment", DEFAULT_PRIMARY_SEGMENT)
SCORING_TEMPERATURE = DEFAULT_SCORING_TEMPERATURE
FX_TO_MODEL_CURRENCY = DEFAULT_FX_TO_MODEL_CURRENCY
SCORING_MIN_DENOMINATOR = DEFAULT_SCORING_MIN_DENOMINATOR

NEAR_DEFAULT_CLUSTER = infer_near_default_cluster_from_artifact(
    artifact,
    segment=SCORING_SEGMENT,
)

segment_artifact = get_segment_artifact(
    artifact,
    segment=SCORING_SEGMENT,
)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("MODEL_PATH:", MODEL_PATH)
print("Scoring segment:", SCORING_SEGMENT)
print("Artifact version:", artifact.get("artifact_version"))
print("Features used by model:", segment_artifact.get("feature_cols"))
print("Cluster labels:", segment_artifact.get("cluster_labels"))
print("Risk rank:", segment_artifact.get("risk_rank"))
print("Inferred near-default cluster:", NEAR_DEFAULT_CLUSTER)

PROJECT_ROOT: D:\users\kamen.dimitrov\desktop\SOFTUNI\AI_and_ML_upskill_program\machine_learning\08_final_project_3
MODEL_PATH: D:\users\kamen.dimitrov\desktop\SOFTUNI\AI_and_ML_upskill_program\machine_learning\08_final_project_3\outputs\saved_models\nonfinancial_credit_scorecard_kmeans_k5_v3.joblib
Scoring segment: Non-financial
Artifact version: v3_scorecard
Features used by model: ['structural_distress_risk', 'earnings_risk', 'operating_cashflow_risk', 'liquidity_risk', 'leverage_risk', 'debt_service_risk']
Cluster labels: {2: '1 - Low risk / investment-grade-like', 1: '2 - Moderate risk / lower-investment-grade-like', 0: '3 - Elevated risk / leveraged', 4: '4 - High risk / speculative', 3: '5 - Distressed / near-default proxy'}
Risk rank: {2: 1, 1: 2, 0: 3, 4: 4, 3: 5}
Inferred near-default cluster: 3


## 2. Manual input example

Edit the numbers below to score a private company, competitor, or simulated case.

Currency note: if your model was trained on USD and the input is in EUR, set `FX_TO_MODEL_CURRENCY` to the EUR/USD conversion rate. Ratios are mostly unaffected; `log_assets` is affected.

In [3]:
manual_input_2025 = pd.DataFrame([{
    "company_name": "Manual 2025 Company",
    "fiscal_year": 2025,
    "currency": "USD",
    "major_sector": "Manufacturing / Industrials",
    "financial_flag": "Non-financial",

    # Values converted from thousand BGN to full USD units at 1.7 BGN/USD
    "revenue": 48_585_294.12,
    "assets": 29_721_275.23,
    "current_assets": 10_037_745.82,
    "cash": 34_804.65,
    "receivables": 2_208_235.29,
    "inventory": 7_794_705.88,
    "equity": 9_082_352.94,
    "current_liabilities": 12_722_352.94,
    "liabilities": 20_638_823.53,
    "long_term_debt": 7_910_588.24,
    "short_term_debt": 1_478_235.29,
    "net_income": 949_411.76,
    "cfo": 3_558_235.29,
    "capex": 588_235.29,
    "operating_income": 1_664_705.88,
    "interest_expense": 597_647.06,

    # Optional EBITDA inputs used by the patched v3 scorer.
    # If direct EBITDA is not available, scoring.py can calculate it from
    # operating_income + depreciation_amortization when both are supplied.
    "depreciation_amortization": 2_713_000,
    "ebitda": np.nan,
}])

template_path = INPUT_PATH / "model_2025_company.csv"

manual_input_2025.to_csv(template_path, index=False)
print("Template saved to:", template_path)


Template saved to: D:\users\kamen.dimitrov\desktop\SOFTUNI\AI_and_ML_upskill_program\machine_learning\08_final_project_3\inputs\model_2025_company.csv


In [4]:
eeec_input_2025 = pd.DataFrame([{
    "company_name": "Eastern European Electric Company B.V.",
    "fiscal_year": 2025,
    "currency": "BGN",
    "major_sector": "Transportation / Utilities",
    "financial_flag": "Non-financial",
    "revenue": 2_572_207_000,
    "assets": 2_127_723_000,
    "current_assets": 1_024_267_000,
    "cash": 224_558_000,
    "receivables": 465_950_000,
    "inventory": 42_958_000,
    "equity": 457_686_000,
    "current_liabilities": 541_002_000,
    "liabilities": 1_670_037_000,
    "long_term_debt": 988_887_000,
    "short_term_debt": 14_753_000,
    "net_income": 36_799_000,
    "cfo": 265_267_000,
    "capex": 212_690_000,
    "operating_income": 54_961_000,
    "interest_expense": 102_302_000,

    # Optional EBITDA inputs used by the patched v3 scorer.
    # If direct EBITDA is not available, scoring.py can calculate it from
    # operating_income + depreciation_amortization when both are supplied.
    "depreciation_amortization": 119_519_000,
    "ebitda": 259_574_000,
}])

template_path = INPUT_PATH / "EEEC_2025.csv"

eeec_input_2025.to_csv(template_path, index=False)
print("Template saved to:", template_path)

Template saved to: D:\users\kamen.dimitrov\desktop\SOFTUNI\AI_and_ML_upskill_program\machine_learning\08_final_project_3\inputs\EEEC_2025.csv


In [5]:
template_path = INPUT_PATH / "model_2025_company.csv"

current_input = pd.read_csv(template_path)
current_input

,company_name,fiscal_year,currency,major_sector,financial_flag,revenue,assets,current_assets,cash,receivables,inventory,equity,current_liabilities,liabilities,long_term_debt,short_term_debt,net_income,cfo,capex,operating_income,interest_expense,depreciation_amortization,ebitda
0,Manual 2025 Company,2025,USD,Manufacturing / Industrials,Non-financial,48585294.12,29721275.23,10037745.82,34804.65,2208235.29,7794705.88,9082352.94,12722352.94,20638823.53,7910588.24,1478235.29,949411.76,3558235.29,588235.29,1664705.88,597647.06,2713000,NaN


In [6]:
scored_manual = score_companies(
    input_df=current_input,
    artifact=artifact,
    segment=SCORING_SEGMENT,
    temperature=SCORING_TEMPERATURE,
    fx_to_model_currency=FX_TO_MODEL_CURRENCY,
    min_denominator=DEFAULT_SCORING_MIN_DENOMINATOR,
    near_default_cluster=NEAR_DEFAULT_CLUSTER,
)

scored_manual_with_outlook = add_adjacent_bucket_distances_and_outlook(
    scored_manual,
    artifact,
    neutral_band=0.15,
    upgrade_boundary_multiplier=1.35,
    downgrade_boundary_multiplier=1.35,
)

current_external_rating = None # optional, none is not existant
scored_manual_with_outlook = apply_credit_guardrails(
    scored_manual_with_outlook,
    external_rating=current_external_rating
)

existing_summary_cols_with_outlook = [
    col for col in SUMMARY_COLS_WITH_OUTLOOK
    if col in scored_manual_with_outlook.columns
]

missing_summary_cols_with_outlook = [
    col for col in SUMMARY_COLS_WITH_OUTLOOK
    if col not in scored_manual_with_outlook.columns
]

if missing_summary_cols_with_outlook:
    print("Missing summary columns:", missing_summary_cols_with_outlook)

display(scored_manual_with_outlook[existing_summary_cols_with_outlook])

,company_name,fiscal_year,assigned_cluster,cluster_label,risk_rank,cluster_affinity,near_default_affinity,distance_to_assigned_cluster,upper_bucket_cluster,upper_bucket_label,distance_to_upper_bucket,lower_bucket_cluster,lower_bucket_label,distance_to_lower_bucket,outlook_flag,outlook_reason,scorecard_credit_score,feature_coverage_pct,warning_flags,guardrail_level,guardrail_flags,guardrail_summary,analyst_interpretation,commercial_conclusion
0,Manual 2025 Company,2025,1,2 - Moderate risk / lower-investment-grade-like,2,0.367302,0.113315,0.46142,2,1 - Low risk / investment-grade-like,0.956363,0,3 - Elevated risk / leveraged,1.624461,Neutral,Company remains materially closer to its assig...,30.006705,1.0,"current_ratio_below_1, quick_ratio_below_0_5",High caution,"current_ratio_below_1, quick_ratio_below_0_5, ...",The company is assigned to 2 - Moderate risk /...,The cluster assignment appears primarily suppo...,"The model output may be directionally useful, ..."


In [7]:
# ---------------------------------------------------------------------
# 8. Company ratio and diagnostic snapshot
# ---------------------------------------------------------------------

ratio_cols = artifact.get("ratio_cols", RATIO_COLS)

existing_ratio_cols = [
    col for col in ratio_cols
    if col in scored_manual.columns
]

missing_ratio_cols = [
    col for col in ratio_cols
    if col not in scored_manual.columns
]

if missing_ratio_cols:
    print("Missing ratio/diagnostic columns:", missing_ratio_cols)

company_ratio_snapshot = scored_manual[
    ["company_name"] + existing_ratio_cols
].copy()

display(company_ratio_snapshot.T.round(4))

,0
company_name,Manual 2025 Company
log_assets,17.207374
liabilities_to_assets,0.694412
debt_to_assets,0.315896
debt_to_equity,1.033744
equity_to_assets,0.305584
cash_to_assets,0.001171
net_income_to_assets,0.031944
cfo_to_assets,0.11972
revenue_to_assets,1.634697


## 3. Compare the company to cluster medians

This helps explain why the company was assigned to a specific profile.

In [8]:
comparison_to_cluster = compare_to_cluster_profile(
    scored_manual.iloc[0],
    artifact,
)

comparison_to_cluster

,company_value,assigned_cluster_median,difference,relative_position
metric,,,,
log_assets,17.2074,22.8058,-5.5984,below_cluster_median
liabilities_to_assets,0.6944,0.6708,0.0236,above_cluster_median
debt_to_assets,0.3159,0.2948,0.0211,above_cluster_median
equity_to_assets,0.3056,0.3013,0.0042,above_cluster_median
current_ratio,0.7890,1.0101,-0.2211,below_cluster_median
quick_ratio,0.1763,0.3506,-0.1743,below_cluster_median
cash_to_assets,0.0012,0.0306,-0.0294,below_cluster_median
net_income_to_assets,0.0319,0.0335,-0.0015,below_cluster_median
cfo_to_assets,0.1197,0.0813,0.0384,above_cluster_median


## 4. Scenario analysis

This section shows how a company migrates across clusters under stress cases

In [9]:
# ---------------------------------------------------------------------
# 10. Build scenario inputs
# ---------------------------------------------------------------------

scenario_input = make_scenarios(current_input.iloc[0])

scenario_input_cols = [
    "scenario",
    "assets",
    "liabilities",
    "equity",
    "cash",
    "revenue",
    "net_income",
    "cfo",
    "long_term_debt",
    "short_term_debt",
    "operating_income",
    "interest_expense",
    "depreciation_amortization",
    "ebitda",
]

existing_scenario_input_cols = [
    col for col in scenario_input_cols
    if col in scenario_input.columns
]

missing_scenario_input_cols = [
    col for col in scenario_input_cols
    if col not in scenario_input.columns
]

if missing_scenario_input_cols:
    print("Missing scenario input columns:", missing_scenario_input_cols)

display(scenario_input[existing_scenario_input_cols])

,scenario,assets,liabilities,equity,cash,revenue,net_income,cfo,long_term_debt,short_term_debt,operating_income,interest_expense,depreciation_amortization,ebitda
0,base,29721275.23,2.063882e+07,9082352.940,34804.650,4.858529e+07,949411.760,3.558235e+06,7.910588e+06,1.478235e+06,1664705.880,597647.06,2713000,NaN
0,revenue_down_15pct,29721275.23,2.063882e+07,9082352.940,34804.650,4.129750e+07,664588.232,2.668676e+06,7.910588e+06,1.478235e+06,1165294.116,597647.06,2713000,NaN
0,debt_up_25pct,29721275.23,2.261647e+07,9082352.940,0.000,4.858529e+07,949411.760,3.558235e+06,9.888235e+06,1.478235e+06,1664705.880,597647.06,2713000,NaN
0,cash_burn_case,29721275.23,2.063882e+07,9082352.940,17402.325,4.858529e+07,-949411.760,-3.558235e+06,7.910588e+06,1.478235e+06,1664705.880,597647.06,2713000,NaN
0,near_default_stress,29721275.23,3.269340e+07,-2972127.523,34804.650,4.858529e+07,949411.760,3.558235e+06,2.229096e+07,4.458191e+06,239058.824,1000000.00,2713000,NaN


In [10]:
# ---------------------------------------------------------------------
# 11. Score scenarios
# ---------------------------------------------------------------------

scored_scenarios = score_companies(
    input_df=scenario_input,
    artifact=artifact,
    segment=SCORING_SEGMENT,
    temperature=SCORING_TEMPERATURE,
    fx_to_model_currency=FX_TO_MODEL_CURRENCY,
    min_denominator=SCORING_MIN_DENOMINATOR,
    near_default_cluster=NEAR_DEFAULT_CLUSTER,
)

scored_scenarios = add_adjacent_bucket_distances_and_outlook(
    scored_scenarios,
    artifact,
    neutral_band=0.15,
    upgrade_boundary_multiplier=1.35,
    downgrade_boundary_multiplier=1.35,
)

scored_scenarios = apply_credit_guardrails(
    scored_scenarios,
    external_rating=current_external_rating,
)

scenario_summary_cols = artifact.get("scenario_summary_cols", SCENARIO_SUMMARY_COLS)

# Add selected diagnostic ratios for direct scenario comparison.
scenario_diagnostic_cols = [
    "liabilities_to_assets",
    "debt_to_assets",
    "debt_to_equity",
    "equity_to_assets",
    "cfo_to_assets",
    "current_ratio",
    "quick_ratio",
    "interest_coverage",
    "fcf_to_debt",
    "ebitda",
    "ebitda_margin",
    "debt_to_ebitda",
    "net_debt_to_ebitda",
    "ebitda_interest_coverage",
    "debt_service_risk",
]

scenario_display_cols = []

for col in list(scenario_summary_cols) + scenario_diagnostic_cols:
    if col in scored_scenarios.columns and col not in scenario_display_cols:
        scenario_display_cols.append(col)

missing_scenario_cols = [
    col for col in list(scenario_summary_cols) + scenario_diagnostic_cols
    if col not in scored_scenarios.columns
]

if missing_scenario_cols:
    print("Missing scenario columns:", missing_scenario_cols)

display(scored_scenarios[scenario_display_cols])

,scenario,assigned_cluster,cluster_label,risk_rank,cluster_affinity,near_default_affinity,distance_to_assigned_cluster,upper_bucket_cluster,upper_bucket_label,distance_to_upper_bucket,lower_bucket_cluster,lower_bucket_label,distance_to_lower_bucket,outlook_flag,outlook_reason,scorecard_credit_score,feature_coverage_pct,warning_flags,guardrail_level,guardrail_flags,guardrail_summary,analyst_interpretation,commercial_conclusion,liabilities_to_assets,debt_to_assets,debt_to_equity,equity_to_assets,cfo_to_assets,current_ratio,quick_ratio,interest_coverage,fcf_to_debt,ebitda,ebitda_margin,debt_to_ebitda,net_debt_to_ebitda,ebitda_interest_coverage,debt_service_risk
0,base,1,2 - Moderate risk / lower-investment-grade-like,2,0.367302,0.113315,0.461420,2,1 - Low risk / investment-grade-like,0.956363,0,3 - Elevated risk / leveraged,1.624461,Neutral,Company remains materially closer to its assig...,30.006705,1.0,"current_ratio_below_1, quick_ratio_below_0_5",High caution,"current_ratio_below_1, quick_ratio_below_0_5, ...",The company is assigned to 2 - Moderate risk /...,The cluster assignment appears primarily suppo...,"The model output may be directionally useful, ...",0.694412,0.315896,1.033744,0.305584,0.11972,0.788985,0.176307,2.785433,0.316334,4377705.880,0.090104,2.144690,2.136740,7.324902,0.046592
1,revenue_down_15pct,1,2 - Moderate risk / lower-investment-grade-like,2,0.372599,0.112475,0.353703,2,1 - Low risk / investment-grade-like,0.910776,0,3 - Elevated risk / leveraged,1.532482,Neutral,Company remains materially closer to its assig...,33.896620,1.0,"current_ratio_below_1, quick_ratio_below_0_5",High caution,"weak_interest_coverage, current_ratio_below_1,...",The company is assigned to 2 - Moderate risk /...,The cluster assignment appears primarily suppo...,"The model output may be directionally useful, ...",0.694412,0.315896,1.033744,0.305584,0.08979,0.788985,0.176307,1.949803,0.221587,3878294.116,0.093911,2.420864,2.411890,6.489272,0.210088
2,debt_up_25pct,1,2 - Moderate risk / lower-investment-grade-like,2,0.368283,0.114273,0.452401,2,1 - Low risk / investment-grade-like,0.958585,0,3 - Elevated risk / leveraged,1.623094,Neutral,Company remains materially closer to its assig...,32.610411,1.0,"current_ratio_below_1, quick_ratio_below_0_5",High caution,"current_ratio_below_1, quick_ratio_below_0_5, ...",The company is assigned to 2 - Moderate risk /...,The cluster assignment appears primarily suppo...,"The model output may be directionally useful, ...",0.760952,0.382435,1.251490,0.305584,0.11972,0.788985,0.173571,2.785433,0.261295,4377705.880,0.090104,2.596445,2.596445,7.324902,0.074827
3,cash_burn_case,4,4 - High risk / speculative,4,0.314235,0.172921,0.551699,0,3 - Elevated risk / leveraged,0.996641,3,5 - Distressed / near-default proxy,1.149008,Neutral,Company remains materially closer to its assig...,63.339858,1.0,"current_ratio_below_1, quick_ratio_below_0_5, ...",High caution,"weak_fcf_to_debt, current_ratio_below_1, quick...",The company is assigned to 4 - High risk / spe...,The cluster assignment appears primarily suppo...,"The model output may be directionally useful, ...",0.694412,0.315896,1.033744,0.305584,-0.11972,0.788985,0.174939,2.785433,-0.441639,4377705.880,0.090104,2.144690,2.140715,7.324902,0.296592
4,near_default_stress,1,2 - Moderate risk / lower-investment-grade-like,2,0.257552,0.248028,1.264633,2,1 - Low risk / investment-grade-like,1.575373,0,3 - Elevated risk / leveraged,2.006955,Positive,Company is closer to the stronger adjacent buc...,62.955405,1.0,"liabilities_exceed_assets, negative_equity, hi...",Override required,"elevated_debt_to_assets, high_debt_to_assets, ...",Structural balance-sheet distress indicators w...,The cluster assignment appears primarily suppo...,Manual analyst override is required. The model...,1.100000,0.900000,-9.000000,-0.100000,0.11972,0.788985,0.176307,0.239059,0.111032,2952058.824,0.060760,9.061184,9.049394,2.952059,0.701845


## 5. Save report outputs to files


In [11]:
# ---------------------------------------------------------------------
# 12. Build and save credit report outputs
# ---------------------------------------------------------------------

report_tables = build_credit_report_tables(
    scored_manual_with_outlook=scored_manual_with_outlook,
    scored_manual=scored_manual,
    comparison_to_cluster=comparison_to_cluster,
    scenario_input=scenario_input,
    scored_scenarios=scored_scenarios,
    artifact=artifact,
)

current_file_name = "model_2025_company"

saved_paths = save_credit_report_outputs(
    tables=report_tables,
    output_path=OUTPUT_PATH,
    base_filename=current_file_name,
)

print("Scores saved to:", saved_paths["score_csv"])
print("Scenario CSV saved to:", saved_paths["scenario_csv"])
print("Full Excel score report saved to:", saved_paths["report_xlsx"])

display(report_tables["scored_file"])
display(report_tables["scenario_file"])

Scores saved to: D:\users\kamen.dimitrov\desktop\SOFTUNI\AI_and_ML_upskill_program\machine_learning\08_final_project_3\outputs\company_scores\model_2025_company_score_result.csv
Scenario CSV saved to: D:\users\kamen.dimitrov\desktop\SOFTUNI\AI_and_ML_upskill_program\machine_learning\08_final_project_3\outputs\company_scores\model_2025_company_scenario_analysis.csv
Full Excel score report saved to: D:\users\kamen.dimitrov\desktop\SOFTUNI\AI_and_ML_upskill_program\machine_learning\08_final_project_3\outputs\company_scores\model_2025_company_score_report.xlsx


,company_name,fiscal_year,assigned_cluster,cluster_label,risk_rank,cluster_affinity,near_default_affinity,distance_to_assigned_cluster,upper_bucket_cluster,upper_bucket_label,distance_to_upper_bucket,lower_bucket_cluster,lower_bucket_label,distance_to_lower_bucket,outlook_flag,outlook_reason,feature_coverage_pct,warning_flags,guardrail_level,guardrail_flags,guardrail_summary,analyst_interpretation,commercial_conclusion
0,Manual 2025 Company,2025,1,2 - Moderate risk / lower-investment-grade-like,2,0.367302,0.113315,0.46142,2,1 - Low risk / investment-grade-like,0.956363,0,3 - Elevated risk / leveraged,1.624461,Neutral,Company remains materially closer to its assig...,1.0,"current_ratio_below_1, quick_ratio_below_0_5",High caution,"current_ratio_below_1, quick_ratio_below_0_5, ...",The company is assigned to 2 - Moderate risk /...,The cluster assignment appears primarily suppo...,"The model output may be directionally useful, ..."


,scenario,assigned_cluster,cluster_label,risk_rank,cluster_affinity,near_default_affinity,distance_to_assigned_cluster,upper_bucket_cluster,upper_bucket_label,distance_to_upper_bucket,lower_bucket_cluster,lower_bucket_label,distance_to_lower_bucket,outlook_flag,outlook_reason,scorecard_credit_score,feature_coverage_pct,warning_flags,guardrail_level,guardrail_flags,guardrail_summary,analyst_interpretation,commercial_conclusion,log_assets,liabilities_to_assets,debt_to_assets,debt_to_equity,equity_to_assets,cash_to_assets,net_income_to_assets,cfo_to_assets,revenue_to_assets,current_ratio,quick_ratio,interest_coverage,fcf_to_debt,operating_margin,gross_margin,cfo_to_liabilities,capex_to_revenue,total_debt,net_debt,ebitda,ebitda_margin,debt_to_ebitda,net_debt_to_ebitda,ebitda_interest_coverage,leverage_risk,liquidity_risk,earnings_risk,operating_cashflow_risk,debt_service_risk,debt_service_risk_legacy,structural_distress_risk
0,base,1,2 - Moderate risk / lower-investment-grade-like,2,0.367302,0.113315,0.461420,2,1 - Low risk / investment-grade-like,0.956363,0,3 - Elevated risk / leveraged,1.624461,Neutral,Company remains materially closer to its assig...,30.006705,1.0,"current_ratio_below_1, quick_ratio_below_0_5",High caution,"current_ratio_below_1, quick_ratio_below_0_5, ...",The company is assigned to 2 - Moderate risk /...,The cluster assignment appears primarily suppo...,"The model output may be directionally useful, ...",17.207374,0.694412,0.315896,1.033744,0.305584,0.001171,0.031944,0.11972,1.634697,0.788985,0.176307,2.785433,0.316334,0.034264,NaN,0.172405,0.012107,9.388824e+06,9.354019e+06,4377705.880,0.090104,2.144690,2.136740,7.324902,0.275204,0.985965,0.180562,0.0,0.046592,0.064370,0
1,revenue_down_15pct,1,2 - Moderate risk / lower-investment-grade-like,2,0.372599,0.112475,0.353703,2,1 - Low risk / investment-grade-like,0.910776,0,3 - Elevated risk / leveraged,1.532482,Neutral,Company remains materially closer to its assig...,33.896620,1.0,"current_ratio_below_1, quick_ratio_below_0_5",High caution,"weak_interest_coverage, current_ratio_below_1,...",The company is assigned to 2 - Moderate risk /...,The cluster assignment appears primarily suppo...,"The model output may be directionally useful, ...",17.207374,0.694412,0.315896,1.033744,0.305584,0.001171,0.022361,0.08979,1.389493,0.788985,0.176307,1.949803,0.221587,0.028217,NaN,0.129304,0.014244,9.388824e+06,9.354019e+06,3878294.116,0.093911,2.420864,2.411890,6.489272,0.275204,0.985965,0.276393,0.0,0.210088,0.315059,0
2,debt_up_25pct,1,2 - Moderate risk / lower-investment-grade-like,2,0.368283,0.114273,0.452401,2,1 - Low risk / investment-grade-like,0.958585,0,3 - Elevated risk / leveraged,1.623094,Neutral,Company remains materially closer to its assig...,32.610411,1.0,"current_ratio_below_1, quick_ratio_below_0_5",High caution,"current_ratio_below_1, quick_ratio_below_0_5, ...",The company is assigned to 2 - Moderate risk /...,The cluster assignment appears primarily suppo...,"The model output may be directionally useful, ...",17.207374,0.760952,0.382435,1.251490,0.305584,0.000000,0.031944,0.11972,1.634697,0.788985,0.173571,2.785433,0.261295,0.034264,NaN,0.157329,0.012107,1.136647e+07,1.136647e+07,4377705.880,0.090104,2.596445,2.596445,7.324902,0.362411,0.985965,0.180562,0.0,0.074827,0.064370,0
3,cash_burn_case,4,4 - High risk / speculative,4,0.314235,0.172921,0.551699,0,3 - Elevated risk / leveraged,0.996641,3,5 - Distressed / near-default proxy,1.149008,Neutral,Company remains materially closer to its assig...,63.339858,1.0,"current_ratio_below_1, quick_ratio_below_0_5, ...",High caution,"weak_fcf_to_debt, current_ratio_below_1, quick...",The company is assigned to 4 - High risk / spe...,The cluster assignment appears primarily suppo...,"The model output may be directionally useful, ...",17.207374,0.694412,0.315896,1.033744,0.305584,0.000586,-0.031944,-0.11972,1.634697,0.788985,0.174939,2.785433,-0.441639,0.034264,NaN,-0.172405,0.012107,9.388824e+06,9.371421e+06,4377705

In [12]:
guardrail_cols = [
    "guardrail_level",
    "guardrail_flags",
    "guardrail_summary",
    "analyst_interpretation",
    "commercial_conclusion",
]

display(scored_manual_with_outlook[guardrail_cols])

,guardrail_level,guardrail_flags,guardrail_summary,analyst_interpretation,commercial_conclusion
0,High caution,"current_ratio_below_1, quick_ratio_below_0_5, ...",The company is assigned to 2 - Moderate risk /...,The cluster assignment appears primarily suppo...,"The model output may be directionally useful, ..."


In [13]:
pdf_path = save_credit_pdf_report(
    report_tables=report_tables,
    scored_manual_with_outlook=scored_manual_with_outlook,
    comparison_to_cluster=comparison_to_cluster,
    scored_scenarios=scored_scenarios,
    artifact=artifact,
    output_path=OUTPUT_PATH,
    base_filename=current_file_name,
)

print("PDF credit report saved to:", pdf_path)

PDF credit report saved to: D:\users\kamen.dimitrov\desktop\SOFTUNI\AI_and_ML_upskill_program\machine_learning\08_final_project_3\outputs\company_scores\model_2025_company_credit_report.pdf


## 6. Optional: inspect saved artifact internals

In [14]:
# ---------------------------------------------------------------------
# Optional artifact inspection
# ---------------------------------------------------------------------

nonfin_artifact = get_segment_artifact(
    artifact,
    segment=SCORING_SEGMENT,
)

print("Artifact version:")
print(artifact.get("artifact_version"))

print("\nPrimary segment:")
print(artifact.get("primary_segment"))

print("\nFeatures used by model:")
print(nonfin_artifact.get("feature_cols"))

print("\nCluster labels:")
print(nonfin_artifact.get("cluster_labels"))

print("\nRisk rank:")
print(nonfin_artifact.get("risk_rank"))

print("\nCluster sizes:")
display(nonfin_artifact.get("cluster_sizes"))

print("\nCluster profile:")
display(nonfin_artifact.get("cluster_profile"))

print("\nCluster profile ranked:")
display(nonfin_artifact.get("cluster_profile_ranked"))

Artifact version:
v3_scorecard

Primary segment:
Non-financial

Features used by model:
['structural_distress_risk', 'earnings_risk', 'operating_cashflow_risk', 'liquidity_risk', 'leverage_risk', 'debt_service_risk']

Cluster labels:
{2: '1 - Low risk / investment-grade-like', 1: '2 - Moderate risk / lower-investment-grade-like', 0: '3 - Elevated risk / leveraged', 4: '4 - High risk / speculative', 3: '5 - Distressed / near-default proxy'}

Risk rank:
{2: 1, 1: 2, 0: 3, 4: 4, 3: 5}

Cluster sizes:


,cluster,issuer_years,issuers
0,0,6593,2032
1,1,4341,1241
2,2,7087,1949
3,3,3350,1572
4,4,2632,1269



Cluster profile:


,scorecard_credit_score,structural_distress_risk,earnings_risk,operating_cashflow_risk,liquidity_risk,leverage_risk,debt_service_risk,liabilities_risk,debt_load_risk,equity_buffer_risk,cash_buffer_risk,current_liquidity_risk,quick_liquidity_risk,profitability_risk,cashflow_risk,coverage_risk,fcf_risk,ebitda_margin_risk,debt_to_ebitda_risk,net_debt_to_ebitda_risk,ebitda_coverage_risk,negative_ebitda_flag,negative_equity_flag,liabilities_exceed_assets_flag,log_assets,small_company_flag,mid_company_flag,large_company_flag,total_debt,net_debt,ebitda,liabilities_to_assets,debt_to_assets,debt_to_equity,equity_to_assets,cash_to_assets,net_income_to_assets,cfo_to_assets,revenue_to_assets,current_ratio,quick_ratio,interest_coverage,fcf_to_debt,operating_margin,gross_margin,cfo_to_liabilities,capex_to_revenue,ebitda_margin,debt_to_ebitda,net_debt_to_ebitda,ebitda_interest_coverage,debt_service_risk_legacy
cluster,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
0,58.333333,0.0,1.000000,1.00000,0.000000,0.039044,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.00000,1.0,1.000000,1.000000,1.000000,0.713506,1.0,1.0,0.0,0.0,18.683641,0.0,0.0,1.0,2.258896e+07,-6.722000e+06,-27983250.0,0.299129,0.126675,0.259263,0.600237,0.277374,-0.258140,-0.198351,0.240246,4.310476,2.130207,-27.678366,-1.230223,-0.818926,0.406889,-0.745634,0.028211,-0.697438,9.780351,3.997271,-23.494511,1.000000
1,32.468255,0.0,0.165341,0.00000,0.737445,0.266252,0.210794,0.401415,0.074700,0.246633,0.771301,0.791910,0.865893,0.165341,0.00000,0.0,0.000000,0.071443,0.343185,0.389702,0.0,0.0,0.0,0.0,22.805765,0.0,0.0,1.0,2.111126e+09,1.506850e+09,707542500.0,0.670778,0.294820,0.858678,0.301347,0.030583,0.033466,0.081283,0.491363,1.010112,0.350580,3.272810,0.202893,0.128044,0.329196,0.123462,0.025668,0.185711,3.372739,2.863957,5.415796,0.102046
2,11.200344,0.0,0.009368,0.00000,0.067104,0.071860,0.087585,0.006860,0.000000,0.000000,0.000000,0.000000,0.000000,0.009368,0.00000,0.0,0.000000,0.259325,0.094019,0.000000,0.0,0.0,0.0,0.0,21.193320,0.0,0.0,1.0,5.373860e+08,2.252195e+08,123142500.0,0.453773,0.235761,0.482906,0.485135,0.117420,0.049063,0.093812,0.610141,2.365987,1.275307,7.003767,0.280442,0.107985,0.376434,0.202783,0.021270,0.148135,2.376074,1.251290,10.197074,0.000000
3,77.597603,1.0,1.000000,1.00000,0.709803,0.693849,1.000000,1.000000,0.243899,1.000000,0.000000,0.922982,0.808580,1.000000,1.00000,1.0,1.000000,1.000000,1.000000,1.000000,1.0,1.0,1.0,1.0,18.060449,0.0,0.0,1.0,3.047650e+07,8.512271e+06,-10741683.0,1.086288,0.396340,-0.805166,-0.209223,0.103684,-0.277459,-0.151918,0.451918,0.846273,0.393565,-7.905898,-0.521353,-0.529271,0.331066,-0.150882,0.011156,-0.459393,9.587202,6.870450,-6.715322,1.000000
4,62.536025,0.0,1.000000,0.80405,0.666973,0.289433,0.904431,0.367144,0.000000,0.259848,0.568291,0.747823,0.798204,1.000000,0.80405,1.0,0.964537,1.000000,1.000000,1.000000,1.0,1.0,0.0,0.0,19.570780,0.0,0.0,1.0,1.402670e+08,6.677050e+07,-4183161.5,0.651929,0.244138,0.837096,0.296061,0.048854,-0.068130,-0.008446,0.458308,1.065221,0.401347,-2.720396,-0.091134,-0.116823,0.295145,-0.029930,0.013599,-0.061367,9.125257,7.786122,-1.439512,0.911562



Cluster profile ranked:


,financial_flag,cluster,issuer_years,issuers,median_log_assets,median_liabilities_to_assets,median_debt_to_assets,median_debt_to_equity,median_equity_to_assets,median_cash_to_assets,median_net_income_to_assets,median_cfo_to_assets,median_revenue_to_assets,median_current_ratio,median_quick_ratio,median_interest_coverage,median_fcf_to_debt,median_operating_margin,median_gross_margin,median_ebitda_margin,median_debt_to_ebitda,median_net_debt_to_ebitda,median_ebitda_interest_coverage,median_leverage_risk,median_liquidity_risk,median_earnings_risk,median_operating_cashflow_risk,median_debt_service_risk,median_structural_distress_risk,median_scorecard_credit_score,rating_style_rank,rating_style_label
2,Non-financial,2,7087,1949,21.193320,0.453773,0.235761,0.482906,0.485135,0.117420,0.049063,0.093812,0.610141,2.365987,1.275307,7.003767,0.280442,0.107985,0.376434,0.148135,2.376074,1.251290,10.197074,0.071860,0.067104,0.009368,0.00000,0.087585,0.0,11.200344,1,1 - Low risk / investment-grade-like
1,Non-financial,1,4341,1241,22.805765,0.670778,0.294820,0.858678,0.301347,0.030583,0.033466,0.081283,0.491363,1.010112,0.350580,3.272810,0.202893,0.128044,0.329196,0.185711,3.372739,2.863957,5.415796,0.266252,0.737445,0.165341,0.00000,0.210794,0.0,32.468255,2,2 - Moderate risk / lower-investment-grade-like
0,Non-financial,0,6593,2032,18.683641,0.299129,0.126675,0.259263,0.600237,0.277374,-0.258140,-0.198351,0.240246,4.310476,2.130207,-27.678366,-1.230223,-0.818926,0.406889,-0.697438,9.780351,3.997271,-23.494511,0.039044,0.000000,1.000000,1.00000,1.000000,0.0,58.333333,3,3 - Elevated risk / leveraged
4,Non-financial,4,2632,1269,19.570780,0.651929,0.244138,0.837096,0.296061,0.048854,-0.068130,-0.008446,0.458308,1.065221,0.401347,-2.720396,-0.091134,-0.116823,0.295145,-0.061367,9.125257,7.786122,-1.439512,0.289433,0.666973,1.000000,0.80405,0.904431,0.0,62.536025,4,4 - High risk / speculative
3,Non-financial,3,3350,1572,18.060449,1.086288,0.396340,-0.805166,-0.209223,0.103684,-0.277459,-0.151918,0.451918,0.846273,0.393565,-7.905898,-0.521353,-0.529271,0.331066,-0.459393,9.587202,6.870450,-6.715322,0.693849,0.709803,1.000000,1.00000,1.000000,1.0,77.597603,5,5 - Distressed / near-default proxy
